# Phase 8 — Kubernetes Security

**What you will learn**
- RBAC: Roles, ClusterRoles, RoleBindings, ClusterRoleBindings
- ServiceAccounts and their tokens
- Pod Security Standards and Pod Security Admission
- Admission controllers, including ValidatingAdmissionPolicy
- securityContext: run as non-root, read-only filesystem, drop capabilities, seccomp
- Network Policies from a security view
- Image scanning, SBOM, image signing and pinned digests
- Runtime security (Falco)
- TLS certificates: the cluster PKI and cert-manager
- Secrets management, least privilege and security auditing

**Time:** about 4–5 hours (do it over 2–3 sessions).

**Prerequisites:** Notebooks 00–07. The kind cluster `learn` is running. Docker Desktop is running.

**How to use this notebook**
- Run cells from top to bottom. Click a cell and press **Shift + Enter**.
- Cells that start with `%%bash` run shell commands.
- Cells that start with `%%writefile` create a file under `labs/08/`.
- Read the text before each cell. It tells you what the command does.
- Answers to quizzes are hidden. Click **Answer** to open them.

## Security in one picture

Every request to Kubernetes passes three gates. Security is about controlling each gate, and then controlling what the Pod can do after it starts.

```
 kubectl / CI / Pod
        |
        v
+-----------------+   1. Authentication  "Who are you?"   (certificate, token, OIDC)
|   API Server    |   2. Authorization   "Are you allowed?" (RBAC)
|                 |   3. Admission       "Is this object OK?" (PSA, policies, webhooks)
+-----------------+
        |
        v
      etcd  (object saved)
        |
        v
   kubelet starts the Pod  ->  4. Runtime controls (securityContext, seccomp, NetworkPolicy)
```

In this notebook we go gate by gate.

## Lab setup

First we check that we are on the **lab cluster**. We never run these labs on a real cluster.
If the context is not `kind-learn`, the cell stops with a message.

In [ ]:
%%bash
ctx=$(kubectl config current-context 2>/dev/null)
if [ "$ctx" != "kind-learn" ]; then
  echo "STOP: current context is '$ctx'. Run: kubectl config use-context kind-learn"
  exit 1
fi
echo "OK: using $ctx"
mkdir -p labs/08
kubectl create namespace lab-08 --dry-run=client -o yaml | kubectl apply -f -
kubectl get nodes

---
# Topic 1 — RBAC (Role-Based Access Control)

### What is it?
RBAC is the **authorization** system of Kubernetes. It answers: *"Can user X do verb Y on resource Z in namespace N?"*

Four objects:

| Object | Scope | What it does |
|---|---|---|
| **Role** | one namespace | A list of permissions (rules) |
| **ClusterRole** | whole cluster | A list of permissions. Can also be reused inside namespaces |
| **RoleBinding** | one namespace | Gives a Role or ClusterRole to a subject, **only in this namespace** |
| **ClusterRoleBinding** | whole cluster | Gives a ClusterRole to a subject in **all namespaces** |

A **subject** is who gets the permission: a User, a Group, or a ServiceAccount.

### Why do we need it?
Without RBAC every user and every Pod could delete anything. One wrong script could delete production.
RBAC lets us give each person and each app **only what it needs** (least privilege).

### How does it work internally?
- RBAC is **allow-only**. There is no "deny" rule. If no rule allows the request, it is denied.
- A rule = `apiGroups` + `resources` + `verbs` (+ optional `resourceNames`).
- The API server checks every binding that matches the subject. If any rule allows the request, it passes.

```
 Subject (User / Group / ServiceAccount)
      |
      |  RoleBinding (namespace: shop)
      v
   Role "pod-reader"  --->  rules: get, list, watch on pods
```

**Verbs** map to HTTP methods: `get`, `list`, `watch` (read), `create`, `update`, `patch`, `delete`, `deletecollection`.

**OCI link:** On OKE, OCI IAM decides *who can get a kubeconfig* and reach the cluster. Inside the cluster, Kubernetes RBAC decides *what they can do*. The OCI user OCID becomes the Kubernetes username. You bind RBAC to that OCID (or to an OCI group OCID).

### Real production scenario
A developer team "payments" needs to see logs and restart Pods in namespace `payments`. They must not read Secrets. They must not touch other namespaces.
Solution: a Role in `payments` with read on pods/logs and delete on pods, plus a RoleBinding to their group. No access to `secrets`.

### Hands-on lab: a read-only role for a ServiceAccount

We create:
1. A ServiceAccount `viewer` (an identity for apps — explained in Topic 2).
2. A Role `pod-reader` that can only read Pods and their logs.
3. A RoleBinding that gives the Role to the ServiceAccount.

In [ ]:
%%writefile labs/08/rbac-pod-reader.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: viewer                 # identity name
  namespace: lab-08
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role                     # namespaced permissions
metadata:
  name: pod-reader
  namespace: lab-08            # the Role only works in this namespace
rules:
  - apiGroups: [""]            # "" = core API group (pods, services, configmaps, secrets)
    resources: ["pods", "pods/log"]   # pods/log is a subresource; needed for 'kubectl logs'
    verbs: ["get", "list", "watch"]   # read only
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding              # connects subject -> role
metadata:
  name: viewer-reads-pods
  namespace: lab-08
subjects:
  - kind: ServiceAccount
    name: viewer
    namespace: lab-08          # ServiceAccount subjects always need a namespace
roleRef:                       # roleRef cannot be changed later; delete and recreate instead
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: pod-reader

Apply the file. Then use `kubectl auth can-i` to **test permissions without logging in** as that identity.
`--as=system:serviceaccount:<namespace>:<name>` means "pretend to be this ServiceAccount" (impersonation). You can do this because you are cluster-admin on kind.

In [ ]:
%%bash
kubectl apply -f labs/08/rbac-pod-reader.yaml
SA=system:serviceaccount:lab-08:viewer
echo "list pods in lab-08:     $(kubectl auth can-i list pods -n lab-08 --as=$SA)"
echo "read pod logs in lab-08: $(kubectl auth can-i get pods/log -n lab-08 --as=$SA)"
echo "delete pods in lab-08:   $(kubectl auth can-i delete pods -n lab-08 --as=$SA)"
echo "list secrets in lab-08:  $(kubectl auth can-i list secrets -n lab-08 --as=$SA)"
echo "list pods in default:    $(kubectl auth can-i list pods -n default --as=$SA)"

**Expected output**
```
list pods in lab-08:     yes
read pod logs in lab-08: yes
delete pods in lab-08:   no
list secrets in lab-08:  no
list pods in default:    no
```
The Role works only in `lab-08`, and only for reading Pods.

### See everything a subject can do
`kubectl auth can-i --list` prints all allowed actions. This is the best quick audit tool.

In [ ]:
%%bash
kubectl auth can-i --list -n lab-08 --as=system:serviceaccount:lab-08:viewer

### ClusterRole + RoleBinding (reuse pattern)
A ClusterRole can be bound with a **RoleBinding**. Then it only applies in that one namespace.
This lets you write a permission set once and reuse it in many namespaces.

Kubernetes ships with default ClusterRoles: `view`, `edit`, `admin`, `cluster-admin`. Let's look.

In [ ]:
%%bash
kubectl get clusterroles view edit admin cluster-admin
echo "--- what 'view' allows (first 25 lines) ---"
kubectl describe clusterrole view | head -25

In [ ]:
%%writefile labs/08/rbac-edit-binding.yaml
apiVersion: v1
kind: ServiceAccount
metadata:
  name: deployer
  namespace: lab-08
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding              # RoleBinding -> permissions only inside lab-08
metadata:
  name: deployer-edit
  namespace: lab-08
subjects:
  - kind: ServiceAccount
    name: deployer
    namespace: lab-08
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole            # reuse the built-in 'edit' ClusterRole
  name: edit

In [ ]:
%%bash
kubectl apply -f labs/08/rbac-edit-binding.yaml
SA=system:serviceaccount:lab-08:deployer
echo "create deployments in lab-08: $(kubectl auth can-i create deployments -n lab-08 --as=$SA)"
echo "create deployments in default: $(kubectl auth can-i create deployments -n default --as=$SA)"
echo "create roles in lab-08 (edit cannot change RBAC): $(kubectl auth can-i create roles -n lab-08 --as=$SA)"

**Expected output**
```
create deployments in lab-08: yes
create deployments in default: no
create roles in lab-08 (edit cannot change RBAC): no
```
`edit` cannot change RBAC. This stops a user from giving themselves more power.

### ClusterRoleBinding — use with care
A ClusterRoleBinding gives access in **every** namespace, including `kube-system`.
Typical correct use: a monitoring tool that must list nodes and pods everywhere (read only).

In [ ]:
%%writefile labs/08/rbac-cluster-reader.yaml
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRole
metadata:
  name: node-reader            # ClusterRoles have no namespace
rules:
  - apiGroups: [""]
    resources: ["nodes"]       # nodes are cluster-scoped; only a ClusterRole can grant them
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: ClusterRoleBinding
metadata:
  name: viewer-reads-nodes
subjects:
  - kind: ServiceAccount
    name: viewer
    namespace: lab-08
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: node-reader

In [ ]:
%%bash
kubectl apply -f labs/08/rbac-cluster-reader.yaml
echo "list nodes: $(kubectl auth can-i list nodes --as=system:serviceaccount:lab-08:viewer)"

### Troubleshooting RBAC

| Symptom | Cause | Fix |
|---|---|---|
| `Error from server (Forbidden): pods is forbidden: User "x" cannot list resource "pods"` | No rule allows it | Read the message: it names user, verb, resource, namespace. Add exactly that rule |
| Works in one namespace, not another | RoleBinding exists only in one namespace | Create a RoleBinding in the other namespace |
| `kubectl logs` forbidden but `get pods` works | Missing subresource `pods/log` | Add `pods/log` to resources |
| `kubectl exec` forbidden | Missing `pods/exec` with verb `create` | Add it only if really needed (exec = shell access) |
| Cannot change roleRef | roleRef is immutable | Delete the binding and create a new one |
| CRD resource forbidden | Wrong `apiGroups` | Use the CRD group, e.g. `monitoring.coreos.com` |

Useful commands:
```bash
kubectl auth can-i <verb> <resource> -n <ns> --as=<user>
kubectl auth can-i --list -n <ns> --as=<user>
kubectl get rolebindings,clusterrolebindings -A -o wide | grep <name>
kubectl api-resources -o wide      # shows API group and verbs of each resource
```

### Lab vs Production

| Area | Lab (this notebook) | Production |
|---|---|---|
| Who is admin | You, cluster-admin | Very few break-glass admins; use groups, not single users |
| Subjects | ServiceAccounts | OIDC / OCI IAM groups for people; one ServiceAccount per app |
| Built-in roles | `edit` is fine | Custom Roles; avoid `cluster-admin` and wildcards `*` |
| Review | Manual | RBAC stored in Git, reviewed in pull requests, audited regularly |
| Dangerous verbs | Not a concern | Restrict `secrets` read, `pods/exec`, `escalate`, `bind`, `impersonate` |

### Check yourself
1. Can a RoleBinding reference a ClusterRole? What is the effect?
2. Is there a "deny" rule in RBAC?
3. Which resource string is needed for `kubectl logs`?
4. Why is `list secrets` dangerous even without `get secrets`?

<details><summary>Answer</summary>

1. Yes. The ClusterRole's rules apply only inside the RoleBinding's namespace.
2. No. RBAC is allow-only. Everything not allowed is denied.
3. `pods/log` with verb `get`.
4. `list` returns full objects, including the secret data. So `list` is as powerful as `get`.
</details>

---
# Topic 2 — ServiceAccounts and tokens

### What is it?
A **ServiceAccount (SA)** is an identity for a **process inside a Pod**. Users are for humans. ServiceAccounts are for apps.
Every namespace has a SA called `default`. If a Pod does not set `serviceAccountName`, it uses `default`.

### Why do we need it?
Apps that talk to the API server (operators, CI runners, backup tools like Commvault, Prometheus) need an identity with limited rights.

### How does it work internally?
- Since Kubernetes 1.24, tokens are **short-lived and bound**. The kubelet asks the API (TokenRequest API) for a token and mounts it into the Pod at
  `/var/run/secrets/kubernetes.io/serviceaccount/token`.
- The token is a signed JWT (JSON Web Token). It expires (default about 1 hour) and the kubelet rotates it.
- The token is bound to the Pod. When the Pod is deleted, the token becomes invalid.
- Old style: a Secret of type `kubernetes.io/service-account-token` with a token that **never expires**. Avoid it, except when a tool really needs a static token.

```
Pod starts -> kubelet -> TokenRequest API -> signed JWT (1h, bound to Pod)
           -> projected volume in the Pod -> app reads token -> calls API server
```

**OCI link:** OKE Workload Identity lets a Pod's ServiceAccount call OCI services (Object Storage, Vault) without storing OCI API keys. An OCI IAM policy names the cluster, namespace and ServiceAccount.

### Real production scenario
A backup tool needs to read all namespaces. The team used `default` SA and gave it cluster-admin. Now **every** Pod in that namespace is cluster-admin.
Fix: create a dedicated SA `backup-sa`, give it only the needed ClusterRole, and set `automountServiceAccountToken: false` on the `default` SA.

### Hands-on lab: look at a token
`kubectl create token` uses the TokenRequest API to make a short-lived token for a SA. We decode the JWT payload to see what is inside.

In [ ]:
%%bash
TOKEN=$(kubectl create token viewer -n lab-08 --duration=10m)
echo "Token (first 40 chars): ${TOKEN:0:40}..."
# A JWT is header.payload.signature. Decode the payload (part 2).
PAYLOAD=$(echo "$TOKEN" | cut -d. -f2 | tr '_-' '/+')
# add base64 padding
while [ $(( ${#PAYLOAD} % 4 )) -ne 0 ]; do PAYLOAD="${PAYLOAD}="; done
echo "$PAYLOAD" | base64 -d 2>/dev/null | python3 -m json.tool

**Expected output (shape)**
```
{
    "aud": ["https://kubernetes.default.svc.cluster.local"],
    "exp": 17xxxxxxxx,            <- expiry time
    "iss": "https://kubernetes.default.svc.cluster.local",
    "kubernetes.io": { "namespace": "lab-08", "serviceaccount": { "name": "viewer", ... } },
    "sub": "system:serviceaccount:lab-08:viewer"
}
```
`sub` is the username RBAC sees.

### Use the token like an app would
We call the API server with the token. The `viewer` SA can list pods but not secrets.

In [ ]:
%%bash
TOKEN=$(kubectl create token viewer -n lab-08 --duration=10m)
SERVER=$(kubectl config view --minify -o jsonpath='{.clusters[0].cluster.server}')
echo "--- list pods (allowed) ---"
curl -sk -H "Authorization: Bearer $TOKEN" "$SERVER/api/v1/namespaces/lab-08/pods" | head -5
echo "--- list secrets (forbidden) ---"
curl -sk -H "Authorization: Bearer $TOKEN" "$SERVER/api/v1/namespaces/lab-08/secrets" | grep -E '"(reason|message)"'

**Expected output**
```
--- list pods (allowed) ---
{
  "kind": "PodList",
  ...
--- list secrets (forbidden) ---
  "message": "secrets is forbidden: User \"system:serviceaccount:lab-08:viewer\" cannot list resource \"secrets\" ...",
  "reason": "Forbidden",
```

### Turn off token mounting when the app does not need it
Most apps (web servers, workers) never call the Kubernetes API. A mounted token is just a risk: if the app is hacked, the attacker gets the token.

In [ ]:
%%writefile labs/08/no-token-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: no-token
  namespace: lab-08
spec:
  automountServiceAccountToken: false   # do not mount the SA token into this Pod
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]

In [ ]:
%%bash
kubectl apply -f labs/08/no-token-pod.yaml
kubectl wait --for=condition=Ready pod/no-token -n lab-08 --timeout=90s
kubectl exec -n lab-08 no-token -- ls /var/run/secrets/kubernetes.io/serviceaccount/ 2>&1 || echo "=> No token mounted. Good."

**Expected output**
```
ls: /var/run/secrets/kubernetes.io/serviceaccount/: No such file or directory
=> No token mounted. Good.
```
You can also set `automountServiceAccountToken: false` on the ServiceAccount itself. Then all Pods using it get no token unless they opt in.

### Troubleshooting ServiceAccounts

| Symptom | Cause | Fix |
|---|---|---|
| App gets 403 Forbidden | SA has no binding, or the Pod uses `default` SA | Check `spec.serviceAccountName`; check `kubectl auth can-i --as=system:serviceaccount:ns:sa` |
| App gets 401 Unauthorized | Token expired; app caches the token forever | Use a current client library (it re-reads the file), or re-read the token file |
| Pod fails: `serviceaccount "x" not found` | SA not created before the Pod | Create the SA first (put it in the same Helm chart) |
| External tool needs a token that never expires | Bound tokens expire | Prefer `kubectl create token --duration`; if a static token is unavoidable, create a `kubernetes.io/service-account-token` Secret and rotate it |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| SA per app | Optional | Always one SA per app; never use `default` for real apps |
| Token mount | Default on | Off unless the app calls the API |
| Static tokens | For testing | Avoid; if needed, store in a vault and rotate |
| Cloud access | — | Workload identity (OKE Workload Identity, EKS IRSA/Pod Identity, AKS/GKE Workload Identity) instead of cloud keys in Secrets |

### Check yourself
1. Which identity does a Pod use if `serviceAccountName` is not set?
2. Where is the token mounted inside the Pod?
3. Why are bound tokens safer than the old Secret tokens?
4. What does `automountServiceAccountToken: false` protect against?

<details><summary>Answer</summary>

1. The `default` ServiceAccount of its namespace.
2. `/var/run/secrets/kubernetes.io/serviceaccount/token`.
3. They expire, rotate, and become invalid when the Pod is deleted. Old tokens never expire.
4. If the app is hacked, the attacker cannot steal a Kubernetes API token from the Pod.
</details>

---
# Topic 3 — Pod Security Standards and Pod Security Admission

### What is it?
**Pod Security Standards (PSS)** are three security levels for Pods, defined by Kubernetes:

| Level | Meaning |
|---|---|
| **privileged** | No limits. For system tools (CNI, storage drivers) |
| **baseline** | Blocks known dangerous settings: privileged containers, hostNetwork, hostPID, hostPath, adding risky capabilities |
| **restricted** | Strong hardening: must run as non-root, drop ALL capabilities, no privilege escalation, seccomp `RuntimeDefault` |

**Pod Security Admission (PSA)** is the built-in admission controller that enforces these levels. You turn it on **per namespace with labels**.

### Why do we need it?
A user who can create Pods can create a privileged Pod with `hostPath: /`. That Pod can read every file on the node and take over the node. PSA blocks this.
(PodSecurityPolicy, the old system, was removed in Kubernetes 1.25.)

### How does it work internally?
Three modes, each set with a namespace label:

| Label | Effect |
|---|---|
| `pod-security.kubernetes.io/enforce: <level>` | Reject Pods that break the level |
| `pod-security.kubernetes.io/warn: <level>` | Allow, but show a warning to the user |
| `pod-security.kubernetes.io/audit: <level>` | Allow, but write an annotation in the audit log |

```
kubectl apply pod ->  API server -> PSA reads namespace labels -> checks Pod spec against level
                                         |-- enforce fails -> REJECT
                                         |-- warn fails    -> allow + warning text
                                         |-- audit fails   -> allow + audit log note
```
Note: `enforce` checks **Pods**. A Deployment is still created, but its ReplicaSet cannot create Pods. `warn` and `audit` also check workload objects like Deployments, so you see the problem early.

### Real production scenario
A platform team wants all application namespaces at `restricted`. They first set `warn: restricted` and `audit: restricted` for two weeks, collect the warnings, fix the Helm charts, then switch to `enforce: restricted`. This avoids breaking deployments on day one.

### Hands-on lab
Create a namespace with `enforce: restricted`, then try a bad Pod and a good Pod.

In [ ]:
%%writefile labs/08/psa-namespace.yaml
apiVersion: v1
kind: Namespace
metadata:
  name: lab-08-restricted
  labels:
    pod-security.kubernetes.io/enforce: restricted        # reject Pods that are not 'restricted'
    pod-security.kubernetes.io/enforce-version: latest    # use rules of the current Kubernetes version
    pod-security.kubernetes.io/warn: restricted           # also print warnings
    pod-security.kubernetes.io/audit: restricted          # and record in audit log

In [ ]:
%%writefile labs/08/psa-bad-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: bad
  namespace: lab-08-restricted
spec:
  containers:
    - name: app
      image: nginx:1.27           # runs as root by default
      securityContext:
        privileged: true          # full access to the node -> forbidden

In [ ]:
%%bash
kubectl apply -f labs/08/psa-namespace.yaml
kubectl apply -f labs/08/psa-bad-pod.yaml 2>&1 || true

**Expected output**
```
Error from server (Forbidden): error when creating "labs/08/psa-bad-pod.yaml": pods "bad" is forbidden:
violates PodSecurity "restricted:latest": privileged (container "app" must not set securityContext.privileged=true),
allowPrivilegeEscalation != false (...), unrestricted capabilities (...), runAsNonRoot != true (...), seccompProfile (...)
```
The message lists every rule the Pod breaks. Use it as a checklist.

In [ ]:
%%writefile labs/08/psa-good-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: good
  namespace: lab-08-restricted
spec:
  securityContext:
    runAsNonRoot: true            # kubelet refuses to start the container as UID 0
    runAsUser: 1000               # run as this user ID
    seccompProfile:
      type: RuntimeDefault        # use the container runtime's default syscall filter
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "id && sleep 3600"]
      securityContext:
        allowPrivilegeEscalation: false   # block setuid tricks (sudo-like escalation)
        capabilities:
          drop: ["ALL"]                   # remove all Linux capabilities

In [ ]:
%%bash
kubectl apply -f labs/08/psa-good-pod.yaml
kubectl wait --for=condition=Ready pod/good -n lab-08-restricted --timeout=90s
kubectl logs -n lab-08-restricted good

**Expected output**
```
pod/good created
pod/good condition met
uid=1000 gid=0(root) groups=0(root)
```

### Dry-run a label change before enforcing it
`--dry-run=server` asks PSA: "what would break if I set this level?" It does not change anything. Very useful before turning on enforce in production.

In [ ]:
%%bash
kubectl label --dry-run=server --overwrite ns lab-08 pod-security.kubernetes.io/enforce=restricted

**Expected output**: warnings for Pods in `lab-08` that would fail, e.g. `Warning: existing pods in namespace "lab-08" violate the new PodSecurity enforce level "restricted:latest"` and a list such as `no-token: allowPrivilegeEscalation != false, ...`.

### Troubleshooting PSA

| Symptom | Cause | Fix |
|---|---|---|
| Deployment exists but has 0 Pods | ReplicaSet blocked by PSA | `kubectl describe rs` or `kubectl get events` shows `violates PodSecurity` |
| `runAsNonRoot` error: `container has runAsNonRoot and image will run as root` | Image's default user is root | Set `runAsUser` to a non-zero UID, or use a non-root image (e.g. `nginx-unprivileged`) |
| System Pods (CNI, CSI) blocked | Namespace label too strict | Keep system namespaces at `privileged`; isolate them |
| Helm chart fails after enforcing | Chart lacks securityContext | Add securityContext values to the chart |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Rollout | Enforce at once | warn + audit first, then enforce |
| App namespaces | restricted | restricted (baseline only with a documented exception) |
| System namespaces | — | privileged, and only platform team can deploy there |
| Version | `latest` | Pin `enforce-version` (e.g. `v1.30`) so upgrades do not surprise you; update deliberately |
| Extra rules | — | Add a policy engine (Kyverno, Gatekeeper, ValidatingAdmissionPolicy) for rules PSA does not cover (e.g. allowed registries) |

### Check yourself
1. Name the three Pod Security levels.
2. How do you enable PSA for a namespace?
3. A Deployment is created but no Pods appear in a `restricted` namespace. Where do you look?
4. How do you test a new level without breaking anything?

<details><summary>Answer</summary>

1. privileged, baseline, restricted.
2. With namespace labels `pod-security.kubernetes.io/enforce|warn|audit: <level>`.
3. `kubectl describe rs <name>` or `kubectl get events -n <ns>` — look for `violates PodSecurity`.
4. Use `warn`/`audit` modes, or `kubectl label --dry-run=server ns <ns> pod-security.kubernetes.io/enforce=<level>`.
</details>

---
# Topic 4 — Admission controllers and ValidatingAdmissionPolicy

### What is it?
An **admission controller** is code in the API server that runs **after** authentication and authorization, and **before** the object is saved in etcd.
There are two kinds:
- **Mutating**: can change the object (e.g. add default values, inject a sidecar).
- **Validating**: can only accept or reject.

### Why do we need it?
RBAC says *who* may create a Deployment. It cannot say *what the Deployment may look like*. Admission control enforces rules like:
- "Images must come from our registry."
- "Every Deployment needs an `owner` label."
- "No `latest` tag."

### How does it work internally?
```
request -> authN -> authZ -> MUTATING admission -> schema validation -> VALIDATING admission -> etcd
                              (built-in + webhooks)                    (built-in + VAP + webhooks)
```
Built-in examples (compiled into the API server): `NamespaceLifecycle`, `LimitRanger`, `ServiceAccount`, `ResourceQuota`, `PodSecurity`, `DefaultStorageClass`.

Ways to add your own rules:

| Option | How | Notes |
|---|---|---|
| **ValidatingAdmissionPolicy (VAP)** | YAML with CEL expressions, runs inside API server | GA since 1.30. No extra Pods. Validation only |
| **MutatingAdmissionPolicy** | CEL mutation in API server | Newer (alpha/beta); check your version |
| **Admission webhooks** | API server calls your HTTPS service | Very flexible; you run and secure the service (see notebook 10) |
| **Kyverno** | Policy engine (webhook) with YAML policies | Validate, mutate, generate, verify image signatures |
| **OPA Gatekeeper** | Policy engine with Rego language | Very powerful, steeper learning |

**CEL** (Common Expression Language) is a small, safe expression language, e.g. `object.spec.replicas <= 5`.

### Real production scenario
An incident happened because someone deployed `image: myapp:latest`. A new image was pushed and Pods restarted with untested code.
The platform team adds a policy: images must have an explicit tag that is not `latest`. Now the mistake is blocked at deploy time.

### Hands-on lab: block `:latest` images with ValidatingAdmissionPolicy
Two objects:
- **ValidatingAdmissionPolicy**: the rule (what to check).
- **ValidatingAdmissionPolicyBinding**: where the rule applies and what to do when it fails.

In [ ]:
%%writefile labs/08/vap-no-latest.yaml
apiVersion: admissionregistration.k8s.io/v1
kind: ValidatingAdmissionPolicy
metadata:
  name: no-latest-tag
spec:
  failurePolicy: Fail                    # if the expression errors, reject (safe default)
  matchConstraints:
    resourceRules:                       # which requests this policy looks at
      - apiGroups: ["apps"]
        apiVersions: ["v1"]
        operations: ["CREATE", "UPDATE"]
        resources: ["deployments"]
  validations:
    - expression: >-
        object.spec.template.spec.containers.all(c,
          c.image.contains(':') && !c.image.endsWith(':latest'))
      # 'all' = every container must pass. Image needs a tag, and the tag is not 'latest'.
      message: "Every container image needs an explicit tag that is not 'latest'."
---
apiVersion: admissionregistration.k8s.io/v1
kind: ValidatingAdmissionPolicyBinding
metadata:
  name: no-latest-tag-lab
spec:
  policyName: no-latest-tag              # which policy
  validationActions: ["Deny"]            # Deny = reject. Other options: Warn, Audit
  matchResources:
    namespaceSelector:                   # apply only to namespaces with this label
      matchLabels:
        policy.learn/no-latest: "true"

In [ ]:
%%bash
kubectl apply -f labs/08/vap-no-latest.yaml
kubectl label ns lab-08 policy.learn/no-latest=true --overwrite
sleep 2   # policies take a moment to load
echo "--- try :latest (should fail) ---"
kubectl create deployment web-latest --image=nginx:latest -n lab-08 2>&1 || true
echo "--- try no tag (should fail) ---"
kubectl create deployment web-notag --image=nginx -n lab-08 2>&1 || true
echo "--- try pinned tag (should work) ---"
kubectl create deployment web-pinned --image=nginx:1.27 -n lab-08

**Expected output**
```
--- try :latest (should fail) ---
error: failed to create deployment: deployments.apps "web-latest" is forbidden: ValidatingAdmissionPolicy 'no-latest-tag'
with binding 'no-latest-tag-lab' denied request: Every container image needs an explicit tag that is not 'latest'.
--- try no tag (should fail) ---
... denied request ...
--- try pinned tag (should work) ---
deployment.apps/web-pinned created
```
Note: a registry with a port like `myreg:5000/app` contains `:` even with no tag. A real policy should check the part after the last `/`. This is a good exercise for you.

### Troubleshooting admission policies

| Symptom | Cause | Fix |
|---|---|---|
| Policy has no effect | Binding missing, or namespace label missing | `kubectl get validatingadmissionpolicybinding`; check `matchResources` |
| All requests rejected with expression error | CEL error + `failurePolicy: Fail` | `kubectl describe validatingadmissionpolicy <name>` shows type-check warnings in status |
| Webhook policy blocks the whole cluster | Webhook service down + `failurePolicy: Fail` | Exclude `kube-system`; run webhook with 2+ replicas; set a short `timeoutSeconds` |
| Can't tell which policy rejected | Many policies | The error message names the policy and binding |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Rollout | `Deny` at once | Start with `validationActions: ["Warn","Audit"]`, then `Deny` |
| Scope | One labeled namespace | All app namespaces; exclude `kube-system` and platform namespaces |
| Tooling | VAP only | VAP for simple rules; Kyverno/Gatekeeper for mutation, image verification, reports |
| Storage | Applied by hand | Policies in Git, tested in CI (e.g. `kyverno test`, `kubectl apply --dry-run=server`) |

### Check yourself
1. In which order do mutating and validating admission run?
2. What is the difference between a ValidatingAdmissionPolicy and its Binding?
3. Why is a webhook with `failurePolicy: Fail` risky?
4. Name two policy engines besides VAP.

<details><summary>Answer</summary>

1. Mutating first, then schema validation, then validating.
2. The Policy holds the rule (CEL). The Binding says where it applies and which action (Deny/Warn/Audit).
3. If the webhook service is down, every matching request is rejected. It can block deployments cluster-wide.
4. Kyverno and OPA Gatekeeper.
</details>

---
# Topic 5 — securityContext (hardening a container)

### What is it?
`securityContext` is a field in the Pod spec (Pod level) and in each container (container level). It sets Linux security options for the process.

| Field | Level | What it does |
|---|---|---|
| `runAsNonRoot: true` | Pod/container | Refuse to start if the process would run as UID 0 |
| `runAsUser` / `runAsGroup` | Pod/container | Run with this user/group ID |
| `fsGroup` | Pod | Group owner for mounted volumes, so a non-root user can write |
| `readOnlyRootFilesystem: true` | container | The container cannot write to its own filesystem |
| `allowPrivilegeEscalation: false` | container | Blocks gaining more rights (setuid binaries) |
| `capabilities.drop: ["ALL"]` | container | Removes Linux capabilities (pieces of root power, e.g. `NET_ADMIN`) |
| `seccompProfile.type: RuntimeDefault` | Pod/container | Filters dangerous system calls |
| `privileged: true` | container | Almost full root on the node — avoid |

### Why do we need it?
If an attacker breaks into your app, securityContext limits the damage. A read-only, non-root process with no capabilities cannot install tools, cannot change binaries, and has a hard time escaping to the node.

### How does it work internally?
The kubelet passes these options to the container runtime (containerd). The runtime uses Linux features: user IDs, **capabilities**, **seccomp** (syscall filter), and mount options (read-only).

### Real production scenario
A web app had a file-upload bug. The attacker uploaded a script and changed the app's own files. With `readOnlyRootFilesystem: true` and only `/tmp` writable (an `emptyDir`), the attacker cannot change the app code. With `runAsNonRoot` they cannot install packages.

### Hands-on lab: a hardened Pod, and proving the limits

In [ ]:
%%writefile labs/08/hardened-pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: hardened
  namespace: lab-08
spec:
  automountServiceAccountToken: false
  securityContext:                       # Pod level: applies to all containers
    runAsNonRoot: true
    runAsUser: 1000
    runAsGroup: 3000
    fsGroup: 2000                        # mounted volumes get group 2000
    seccompProfile:
      type: RuntimeDefault
  containers:
    - name: app
      image: busybox:1.36
      command: ["sh", "-c", "sleep 3600"]
      securityContext:                   # container level: overrides Pod level if both set
        readOnlyRootFilesystem: true     # root filesystem is read-only
        allowPrivilegeEscalation: false
        capabilities:
          drop: ["ALL"]
      resources:
        requests: { cpu: 10m, memory: 16Mi }
        limits:   { memory: 32Mi }
      volumeMounts:
        - name: tmp
          mountPath: /tmp                # the only writable place
  volumes:
    - name: tmp
      emptyDir: {}                       # temporary dir, deleted with the Pod

In [ ]:
%%bash
kubectl apply -f labs/08/hardened-pod.yaml
kubectl wait --for=condition=Ready pod/hardened -n lab-08 --timeout=90s
echo "--- who am I? ---"
kubectl exec -n lab-08 hardened -- id
echo "--- write to root filesystem (should fail) ---"
kubectl exec -n lab-08 hardened -- touch /evil 2>&1 || true
echo "--- write to /tmp (should work) ---"
kubectl exec -n lab-08 hardened -- sh -c 'touch /tmp/ok && ls -l /tmp/ok'
echo "--- capabilities of the process ---"
kubectl exec -n lab-08 hardened -- grep Cap /proc/1/status

**Expected output**
```
--- who am I? ---
uid=1000 gid=3000 groups=2000,3000
--- write to root filesystem (should fail) ---
touch: /evil: Read-only file system
--- write to /tmp (should work) ---
-rw-r--r--    1 1000     2000   0 ... /tmp/ok
--- capabilities of the process ---
CapInh: 0000000000000000
CapPrm: 0000000000000000
CapEff: 0000000000000000      <- no capabilities at all
CapBnd: 0000000000000000
CapAmb: 0000000000000000
```

### When an app needs one capability
Example: an app must bind to port 80 as non-root. Drop all, add back only `NET_BIND_SERVICE`. (Better: listen on 8080 and let the Service map port 80 → 8080.)
```yaml
securityContext:
  capabilities:
    drop: ["ALL"]
    add: ["NET_BIND_SERVICE"]   # allowed in 'restricted' level
```

### Troubleshooting securityContext

| Symptom | Cause | Fix |
|---|---|---|
| `CreateContainerConfigError: container has runAsNonRoot and image will run as root` | Image user is root and no `runAsUser` | Set `runAsUser`, or use a non-root image |
| App crashes: `Read-only file system` | App writes to cache/pid/log dirs | Mount `emptyDir` at those paths (`/tmp`, `/var/cache/nginx`, ...) |
| `Permission denied` writing to a volume | Volume owned by root | Set `fsGroup` |
| nginx fails: `bind() to 0.0.0.0:80 failed (13: Permission denied)` | Non-root cannot bind < 1024 | Use port 8080 (`nginx-unprivileged`) or add `NET_BIND_SERVICE` |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Defaults | Set by hand per Pod | Set in Helm chart defaults; enforced by PSA `restricted` |
| Images | busybox | Distroless or minimal non-root images |
| Exceptions | — | Documented, reviewed, limited to a namespace |
| Extra | — | AppArmor/SELinux profiles where available; custom seccomp profiles for sensitive apps |

### Check yourself
1. What is the difference between `runAsNonRoot` and `runAsUser`?
2. An app needs to write temp files but you want a read-only root filesystem. What do you do?
3. What does `capabilities.drop: ["ALL"]` remove?
4. Which setting fixes "Permission denied" on a mounted volume for a non-root user?

<details><summary>Answer</summary>

1. `runAsNonRoot` is a check (refuse UID 0). `runAsUser` sets the actual UID.
2. Mount an `emptyDir` volume at the temp path (e.g. `/tmp`).
3. All Linux capabilities: the pieces of root power like changing network config or file ownership.
4. `fsGroup` at Pod level.
</details>

---
# Topic 6 — Network Policies (security view)

Notebook 04 covered how NetworkPolicy works. Here we focus on **security**: a "default deny" baseline and **egress** control.

### Why it matters for security
By default every Pod can talk to every Pod in the cluster. If one Pod is hacked, the attacker can reach your database directly (this is called **lateral movement**).

### Pattern
1. Default deny all ingress and egress in the namespace.
2. Allow DNS egress (otherwise nothing can resolve names).
3. Allow only the needed flows (frontend → backend → database).

**Note for kind:** kind v0.24+ enforces NetworkPolicy with its default CNI (kindnet). If your policies seem to do nothing, check `kind version`; older versions need Calico or Cilium.

In [ ]:
%%writefile labs/08/netpol-default-deny.yaml
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: default-deny-all
  namespace: lab-08
spec:
  podSelector: {}                 # {} = every Pod in the namespace
  policyTypes: ["Ingress", "Egress"]   # no rules listed = deny all in both directions
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: allow-dns-egress
  namespace: lab-08
spec:
  podSelector: {}
  policyTypes: ["Egress"]
  egress:
    - to:
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: kube-system   # CoreDNS lives here
          podSelector:
            matchLabels:
              k8s-app: kube-dns
      ports:
        - { protocol: UDP, port: 53 }
        - { protocol: TCP, port: 53 }

In [ ]:
%%bash
kubectl apply -f labs/08/netpol-default-deny.yaml
kubectl run nettest -n lab-08 --image=busybox:1.36 --restart=Never -- sleep 3600
kubectl wait --for=condition=Ready pod/nettest -n lab-08 --timeout=90s
echo "--- DNS (allowed) ---"
kubectl exec -n lab-08 nettest -- nslookup kubernetes.default.svc.cluster.local 2>&1 | tail -3
echo "--- HTTP to the API service (blocked by default deny) ---"
kubectl exec -n lab-08 nettest -- wget -T 3 -qO- https://kubernetes.default.svc 2>&1 | tail -1 || true

**Expected output**
```
--- DNS (allowed) ---
Name:	kubernetes.default.svc.cluster.local
Address: 10.96.0.1
--- HTTP to the API service (blocked by default deny) ---
wget: download timed out
```
DNS works because we allowed it. Everything else is blocked.

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Baseline | Manual | Default-deny in every app namespace, created by the namespace template |
| Egress to internet | Not tested | Allow only known destinations; use an egress gateway or OCI NAT + security lists/NSGs |
| CNI | kindnet | Calico/Cilium (Cilium adds L7 and DNS-name policies); on OKE: Flannel has no NetworkPolicy — use OCI VCN-Native CNI + Calico or Cilium |
| Layers | Only NetworkPolicy | NetworkPolicy + OCI NSGs/security lists (defense in depth) |

### Check yourself
1. What does `podSelector: {}` with `policyTypes: [Ingress, Egress]` and no rules do?
2. Why must you allow DNS after a default deny egress?
3. What is lateral movement?

<details><summary>Answer</summary>

1. Denies all incoming and outgoing traffic for every Pod in the namespace.
2. Pods resolve service names through CoreDNS. Without DNS, nothing can find anything.
3. An attacker moving from one hacked system to other systems inside the network.
</details>

---
# Topic 7 — Image security, scanning and supply chain

### What is it?
- **Image scanning**: checking an image for known vulnerabilities (CVEs) in its OS packages and libraries.
- **SBOM** (Software Bill of Materials): a list of every package inside an image. Like an ingredient list.
- **Image signing**: a cryptographic signature that proves *who* built the image and that it was not changed. Tool: **cosign** (Sigstore project).
- **Pinned digest**: referring to an image by its content hash `image@sha256:...` instead of a tag. Tags can move. Digests cannot.
- **Supply-chain security**: protecting every step from source code to running container (code, build, registry, deploy).

### Why do we need it?
- Old base images contain known holes. Attackers scan the internet for them.
- An attacker who can push to your registry can replace `app:1.4` with a bad image. Signing + digests stop this.
- When a new big CVE appears (like Log4Shell), an SBOM tells you in minutes which images contain the library.

### How does it work internally?
```
 code -> CI build -> image -> scan (trivy) -> SBOM -> sign (cosign) -> push to registry
                                                                             |
 cluster:  admission policy (Kyverno / OCI image signing policy) verifies signature + digest -> run
```

**OCI link:** OCIR (OCI Registry) can scan images (Vulnerability Scanning Service). OKE supports **image verification policies**: only images signed with keys from OCI Vault can run.

### Real production scenario
Security reports a critical CVE in OpenSSL. Without SBOMs the team checks 60 images by hand. With SBOMs stored for every build, one search lists the 7 affected images. The team rebuilds them and the CI scan gate stops any new image with that CVE.

### Hands-on lab 1: scan an image with trivy
**trivy** is a free scanner from Aqua Security. Install it with Homebrew (one time). The first scan downloads a vulnerability database (needs internet, ~1 minute).

In [ ]:
%%bash
command -v trivy >/dev/null || brew install trivy
trivy --version | head -1

Scan an **old** image and a **newer** image. `--severity HIGH,CRITICAL` shows only serious findings. `--quiet` hides progress output.

In [ ]:
%%bash
echo "=== old image: nginx:1.20 ==="
trivy image --quiet --severity HIGH,CRITICAL nginx:1.20 2>/dev/null | grep -E "^Total|nginx:1.20" | head -5
echo "=== newer image: nginx:1.27-alpine ==="
trivy image --quiet --severity HIGH,CRITICAL nginx:1.27-alpine 2>/dev/null | grep -E "^Total|alpine" | head -5

**Expected output (numbers change over time)**
```
=== old image: nginx:1.20 ===
nginx:1.20 (debian 11.x)
Total: 1xx (HIGH: 1xx, CRITICAL: 2x)
=== newer image: nginx:1.27-alpine ===
nginx:1.27-alpine (alpine 3.x)
Total: 0 (HIGH: 0, CRITICAL: 0)      <- or a very small number
```
Lesson: small, current base images have far fewer vulnerabilities.

### Use the scan as a CI gate
`--exit-code 1` makes trivy fail the pipeline if it finds issues. This is how you block bad images in Azure DevOps, Jenkins or GitHub Actions.

In [ ]:
%%bash
trivy image --quiet --severity CRITICAL --exit-code 1 nginx:1.20 >/dev/null 2>&1
echo "exit code for nginx:1.20 = $?   (1 = pipeline would FAIL)"

### Hands-on lab 2: create an SBOM
CycloneDX is a standard SBOM format. Then we search it for a package.

In [ ]:
%%bash
trivy image --quiet --format cyclonedx --output labs/08/sbom-nginx.json nginx:1.27-alpine 2>/dev/null
python3 -c "
import json
d = json.load(open('labs/08/sbom-nginx.json'))
comps = d.get('components', [])
print('components in SBOM:', len(comps))
for c in comps:
    if 'openssl' in c.get('name','') or 'libssl' in c.get('name',''):
        print(' ', c['name'], c.get('version'))
"

**Expected output (versions change)**
```
components in SBOM: 6x
  libssl3 3.x.x-rx
```

### Hands-on lab 3: digests — what is really running?
A tag is a pointer. The digest is the real content. Kubernetes records the digest it pulled in `imageID`.

In [ ]:
%%bash
kubectl get pod hardened -n lab-08 -o jsonpath='{.spec.containers[0].image}{"\n"}{.status.containerStatuses[0].imageID}{"\n"}'

**Expected output**
```
busybox:1.36
docker.io/library/busybox@sha256:7c3c3cea5d4d6133d6a694d23382f6a7b32652f23855abdba3eb039ca5995447   <- example
```
In production, deploy with `image: registry/app@sha256:...` (or `app:1.4@sha256:...`). Then nobody can swap the image behind the tag.

### Hands-on lab 4: verify a signed image with cosign
Kubernetes project images are signed with Sigstore "keyless" signing. We verify one. Install cosign first.
The command checks: the signature is valid, and it was made by the Kubernetes release identity.

In [ ]:
%%bash
command -v cosign >/dev/null || brew install cosign
cosign verify registry.k8s.io/pause:3.9 \
  --certificate-identity krel-trust@k8s-releng-prod.iam.gserviceaccount.com \
  --certificate-oidc-issuer https://accounts.google.com 2>&1 | head -5

**Expected output**
```
Verification for registry.k8s.io/pause:3.9 --
The following checks were performed on each of these signatures:
  - The cosign claims were validated
  - Existence of the claims in the transparency log was verified offline
  - The code-signing certificate was verified using trusted certificate authority certificates
```
**How you would sign your own image (in CI, not run here):**
```bash
cosign generate-key-pair                       # creates cosign.key / cosign.pub (store key in a vault)
cosign sign --key cosign.key myregistry/app@sha256:<digest>
cosign verify --key cosign.pub myregistry/app@sha256:<digest>
```
Then a Kyverno `verifyImages` policy (or OKE image verification) rejects unsigned images.

### Troubleshooting image security

| Symptom | Cause | Fix |
|---|---|---|
| Too many CVEs, team ignores them | Fat base images, no priority | Use minimal/distroless images; gate only on CRITICAL/HIGH with fix available (`--ignore-unfixed`) |
| CI gate blocks urgent fix | New CVE with no fix | Time-limited exception in `.trivyignore` with reason and owner |
| Signed image rejected | Signed by tag, deployed by digest (or the other way) | Sign and deploy the **digest** |
| `ImagePullBackOff` on private registry | Missing pull secret | `imagePullSecrets` or node/workload identity to the registry |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Scanning | Manual trivy | In CI on every build + continuous registry scanning (OCIR scanning) |
| Tags | `1.36` | Immutable tags + digests; registry tag immutability on |
| Signing | Verify a public image | Sign in CI; verify at admission (Kyverno/OKE image policy) |
| SBOM | One file | Generated per build, stored with the image (attestation) |
| Registry | Docker Hub | Private registry; only allowed registries via admission policy |

### Check yourself
1. Why is a digest safer than a tag?
2. What is an SBOM useful for during a new CVE?
3. Which trivy flag makes CI fail on findings?
4. What does cosign prove?

<details><summary>Answer</summary>

1. A tag can be moved to another image. A digest is the hash of the content and cannot change.
2. To quickly find which images contain the vulnerable package.
3. `--exit-code 1`.
4. Who signed the image and that it was not changed after signing.
</details>

---
# Topic 8 — Runtime security (Falco)

### What is it?
**Runtime security** watches what containers *do* while running. **Falco** (a CNCF project) reads Linux system calls (with eBPF) and raises alerts on suspicious behavior.

### Why do we need it?
Scanning finds *known* holes before deploy. It cannot see an attack that is happening now. Falco can see, for example:
- A shell started inside a container (`kubectl exec` or a reverse shell).
- A process reading `/etc/shadow`.
- A container writing to `/bin`.
- Unexpected outbound connections.

### How does it work internally?
```
container process -> syscall (open, execve, connect) -> kernel
                                                          |
                                         eBPF probe (Falco) reads event
                                                          |
                                Falco rules engine: "shell in container?" -> ALERT
                                                          |
                         stdout / Falcosidekick -> Slack, Alertmanager, SIEM
```
Example rule (simplified):
```yaml
- rule: Terminal shell in container
  condition: spawned_process and container and proc.name in (bash, sh) and proc.tty != 0
  output: "Shell in container (user=%user.name pod=%k8s.pod.name cmd=%proc.cmdline)"
  priority: NOTICE
```

### Real production scenario
At 02:00 Falco alerts: `Shell in container pod=payments-api-7d...`. Nobody on the team ran `kubectl exec`. The audit log shows a stolen CI token was used. The team revokes the token, isolates the Pod with a NetworkPolicy, and keeps it for forensics.

### Lab (optional)
Falco runs as a DaemonSet with Helm. On kind on macOS it usually works with the `modern_ebpf` driver, but it depends on the Docker Desktop kernel. Try it only if you have time:
```bash
helm repo add falcosecurity https://falcosecurity.github.io/charts
helm install falco falcosecurity/falco -n falco --create-namespace --set driver.kind=modern_ebpf --set tty=true
kubectl logs -n falco -l app.kubernetes.io/name=falco -f     # then 'kubectl exec' into any pod in another terminal
helm uninstall falco -n falco
```

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Rules | Defaults | Tuned to reduce noise; custom rules for your apps |
| Alerts | Logs | Falcosidekick -> Alertmanager / Slack / SIEM, with on-call runbook |
| Response | — | Automated: label/isolate Pod; manual forensics |

### Check yourself
1. What does Falco watch?
2. Why is scanning not enough?

<details><summary>Answer</summary>

1. System calls made by processes in containers (via eBPF), matched against rules.
2. Scanning finds known vulnerabilities before deploy. It cannot detect an active attack or unknown (zero-day) exploits at runtime.
</details>

---
# Topic 9 — TLS certificates: cluster PKI and cert-manager

### What is it?
- **PKI** (Public Key Infrastructure): certificate authorities (CAs) and certificates that prove identity and encrypt traffic.
- Kubernetes itself uses TLS everywhere: kubectl ↔ API server, API server ↔ etcd, API server ↔ kubelet.
- **cert-manager** is a controller that creates and renews certificates for **your apps** (Ingress, internal mTLS) from issuers like Let's Encrypt, a private CA, or a vault.

### Why do we need it?
- Expired certificates are one of the most common causes of outages ("certificate has expired").
- On self-managed clusters (kubeadm), control-plane certs expire after **1 year** unless renewed (upgrades renew them).
- cert-manager renews app certificates automatically before they expire.

### How does it work internally? (cluster PKI)
```
              Cluster CA (ca.crt)  -- signs -->  apiserver.crt, apiserver-kubelet-client.crt, ...
              etcd CA            -- signs -->  etcd server / peer certs
              front-proxy CA     -- signs -->  aggregation layer certs
kubeconfig user cert is signed by the Cluster CA. ServiceAccount tokens are signed by sa.key.
```
On kind (built with kubeadm) the files live in `/etc/kubernetes/pki` inside the control-plane container.
On managed clusters (OKE, EKS, AKS, GKE) the provider manages control-plane certificates.

In [ ]:
%%bash
docker exec learn-control-plane ls /etc/kubernetes/pki
echo "--- expiry of control-plane certificates ---"
docker exec learn-control-plane kubeadm certs check-expiration 2>/dev/null | head -15

**Expected output**
```
apiserver.crt  apiserver.key  apiserver-etcd-client.crt ... ca.crt ca.key ... etcd  front-proxy-ca.crt ... sa.key  sa.pub
--- expiry of control-plane certificates ---
CERTIFICATE                EXPIRES                  RESIDUAL TIME   CERTIFICATE AUTHORITY   EXTERNALLY MANAGED
admin.conf                 <date next year>         364d            ca                      no
apiserver                  <date next year>         364d            ca                      no
...
```

### How cert-manager works
```
  Issuer / ClusterIssuer  (who signs: self-signed, CA, ACME/Let's Encrypt, Vault)
          ^
          |  referenced by
  Certificate  (what you want: DNS names, duration, renewBefore, secretName)
          |
  cert-manager controller -> creates CertificateRequest -> Issuer signs
          |
  Secret (type kubernetes.io/tls: tls.crt, tls.key, ca.crt)  -> used by Ingress / Pods
          |
  renews automatically before expiry (default: at 2/3 of lifetime)
```

### Real production scenario
An internal API certificate was created by hand two years ago. It expired on a Friday night and every client failed. After the incident, the team moved all certificates to cert-manager with alerts on `certmanager_certificate_expiration_timestamp_seconds` in Prometheus.

### Hands-on lab: install cert-manager and issue a certificate
We install cert-manager from its official release manifest (needs internet). `kubectl wait` waits until its Pods are ready.

In [ ]:
%%bash
kubectl apply -f https://github.com/cert-manager/cert-manager/releases/download/v1.15.3/cert-manager.yaml >/dev/null
kubectl wait --for=condition=Available deployment --all -n cert-manager --timeout=180s
kubectl get pods -n cert-manager

We build a small private CA:
1. A **self-signed Issuer** (bootstrap only).
2. A **CA Certificate** signed by it (`isCA: true`).
3. A **CA Issuer** that uses the CA's secret.
4. A **leaf Certificate** for our app, signed by the CA Issuer.

This is the same chain you would use for internal services.

In [ ]:
%%writefile labs/08/cert-manager-ca.yaml
apiVersion: cert-manager.io/v1
kind: Issuer
metadata:
  name: selfsigned-bootstrap
  namespace: lab-08
spec:
  selfSigned: {}                    # signs a cert with its own key (only to create our CA)
---
apiVersion: cert-manager.io/v1
kind: Certificate
metadata:
  name: lab-ca
  namespace: lab-08
spec:
  isCA: true                        # this certificate can sign other certificates
  commonName: lab-08-internal-ca
  secretName: lab-ca-secret         # CA key + cert stored here
  duration: 8760h                   # 1 year
  privateKey:
    algorithm: ECDSA
    size: 256
  issuerRef:
    name: selfsigned-bootstrap
    kind: Issuer
---
apiVersion: cert-manager.io/v1
kind: Issuer
metadata:
  name: lab-ca-issuer
  namespace: lab-08
spec:
  ca:
    secretName: lab-ca-secret       # sign with our CA
---
apiVersion: cert-manager.io/v1
kind: Certificate
metadata:
  name: api-tls
  namespace: lab-08
spec:
  secretName: api-tls               # Secret (type kubernetes.io/tls) that apps / Ingress use
  duration: 2160h                   # 90 days
  renewBefore: 360h                 # renew 15 days before expiry
  dnsNames:                         # names the certificate is valid for
    - api.lab-08.svc
    - api.lab-08.svc.cluster.local
  privateKey:
    algorithm: ECDSA
    size: 256
    rotationPolicy: Always          # new key on every renewal
  issuerRef:
    name: lab-ca-issuer
    kind: Issuer

In [ ]:
%%bash
kubectl apply -f labs/08/cert-manager-ca.yaml
kubectl wait --for=condition=Ready certificate/api-tls -n lab-08 --timeout=120s
kubectl get certificate,issuer -n lab-08
echo "--- decode the leaf certificate ---"
kubectl get secret api-tls -n lab-08 -o jsonpath='{.data.tls\.crt}' | base64 -d \
  | openssl x509 -noout -subject -issuer -dates -ext subjectAltName

**Expected output**
```
NAME                                  READY   SECRET          AGE
certificate.cert-manager.io/api-tls   True    api-tls         10s
certificate.cert-manager.io/lab-ca    True    lab-ca-secret   12s
...
--- decode the leaf certificate ---
subject=
issuer=CN=lab-08-internal-ca
notBefore=...
notAfter=...  (90 days later)
X509v3 Subject Alternative Name:
    DNS:api.lab-08.svc, DNS:api.lab-08.svc.cluster.local
```

### Troubleshooting certificates

| Symptom | Cause | Fix |
|---|---|---|
| Certificate `READY=False` | Issuer not ready, wrong issuerRef, ACME challenge failing | `kubectl describe certificate`, then `kubectl get certificaterequest,order,challenge -n <ns>` |
| `x509: certificate signed by unknown authority` | Client does not trust the CA | Give clients the CA (`ca.crt`); use trust-manager to distribute |
| `x509: certificate is valid for X, not Y` | Hostname not in SANs | Add the name to `dnsNames` |
| `certificate has expired` on kubectl | kubeadm certs not renewed | `kubeadm certs renew all` on control planes, restart static Pods; managed clusters: provider handles |
| cert-manager webhook errors on apply | cert-manager not ready yet | Wait for deployments; check `kubectl get pods -n cert-manager` |

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Issuer | Self-signed CA in a Secret | Let's Encrypt (public) or enterprise CA / Vault / OCI Certificates service (private) |
| Scope | Namespaced Issuer | ClusterIssuer managed by platform team |
| Monitoring | Manual | Prometheus alerts on expiry; dashboards |
| CA key | Plain Secret | HSM/Vault-backed; encryption at rest |
| Control plane | kind kubeadm certs | Managed by provider (OKE), or automated renewal for self-managed |

### Check yourself
1. Where are kubeadm control-plane certificates stored?
2. What are the roles of Issuer and Certificate in cert-manager?
3. What field controls when cert-manager renews?
4. Error: `certificate is valid for a.example.com, not b.example.com`. Fix?

<details><summary>Answer</summary>

1. `/etc/kubernetes/pki` on control-plane nodes.
2. Issuer = who signs. Certificate = the request (names, duration) and where to store the result.
3. `renewBefore` (or default 2/3 of the lifetime).
4. Add `b.example.com` to `dnsNames` in the Certificate.
</details>

---
# Topic 10 — Secrets management and least privilege

### Secrets (short recap — details in notebook 06)
- A Kubernetes Secret is **base64-encoded, not encrypted**. Anyone with `get`/`list` on secrets can read it.
- Protect secrets with: RBAC (very few can read), **encryption at rest** in etcd (KMS provider; on OKE use OCI Vault keys), and an external secret store (OCI Vault, HashiCorp Vault) synced with **External Secrets Operator** or mounted with the **Secrets Store CSI Driver**.

### Least privilege — the principle
Give every identity **only** the access it needs, **only** where it needs it, **only** for as long as it needs it.

Checklist for every app:
- [ ] Own ServiceAccount; token mount off unless needed
- [ ] Role in its own namespace, no wildcards (`*`)
- [ ] No read on Secrets unless needed (and then `resourceNames` for specific secrets)
- [ ] securityContext: non-root, read-only root fs, drop ALL capabilities
- [ ] NetworkPolicy: default deny + needed flows only
- [ ] Image from private registry, pinned digest, scanned
- [ ] Cloud access via workload identity, not static keys

### Dangerous permissions to watch

| Permission | Why dangerous |
|---|---|
| `create pods` in a namespace | Can mount any Secret or SA in that namespace into a Pod |
| `pods/exec` | Shell in running containers |
| `get/list secrets` | Read all credentials |
| `escalate`, `bind` on roles | Give yourself more power |
| `impersonate` | Act as any user/group |
| `nodes/proxy` | Talk to kubelet API directly |
| Any `*` (wildcard) | Includes all of the above, plus future resources |

### Hands-on lab: limit a secret reader to ONE secret
`resourceNames` restricts a rule to named objects. Note: `list` and `watch` cannot be limited by name in a useful way, so we give only `get`.

In [ ]:
%%writefile labs/08/rbac-one-secret.yaml
apiVersion: v1
kind: Secret
metadata:
  name: app-db-password
  namespace: lab-08
stringData:
  password: "lab-only-not-real"
---
apiVersion: v1
kind: Secret
metadata:
  name: other-team-secret
  namespace: lab-08
stringData:
  token: "lab-only-not-real"
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: read-app-db-secret
  namespace: lab-08
rules:
  - apiGroups: [""]
    resources: ["secrets"]
    resourceNames: ["app-db-password"]   # only this one Secret
    verbs: ["get"]                       # no list/watch: they would expose all secrets
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: viewer-read-app-db-secret
  namespace: lab-08
subjects:
  - kind: ServiceAccount
    name: viewer
    namespace: lab-08
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: read-app-db-secret

In [ ]:
%%bash
kubectl apply -f labs/08/rbac-one-secret.yaml
SA=system:serviceaccount:lab-08:viewer
echo "get app-db-password:   $(kubectl auth can-i get secret/app-db-password -n lab-08 --as=$SA)"
echo "get other-team-secret: $(kubectl auth can-i get secret/other-team-secret -n lab-08 --as=$SA)"
echo "list secrets:          $(kubectl auth can-i list secrets -n lab-08 --as=$SA)"

**Expected output**
```
get app-db-password:   yes
get other-team-secret: no
list secrets:          no
```

### Check yourself
1. Is a Kubernetes Secret encrypted by default?
2. Why can `create pods` leak secrets?
3. What does `resourceNames` do?

<details><summary>Answer</summary>

1. No. It is base64-encoded. Encryption at rest must be configured (KMS); managed providers often offer it (OKE with OCI Vault keys).
2. A user can create a Pod that mounts any Secret in the namespace and print it.
3. Limits a rule to specific object names.
</details>

---
# Topic 11 — Security auditing

### What is it?
Auditing means: **record who did what**, and **regularly check** the cluster against a security standard.

Three tools:
1. **Kubernetes audit log** — the API server writes a log entry for each request (who, what, when, result).
2. **RBAC review** — find over-powerful identities (`kubectl auth can-i --list`, list cluster-admin bindings).
3. **CIS benchmark scan** — **kube-bench** checks node and control-plane settings against the CIS Kubernetes Benchmark.

### How the audit log works
An **audit policy** decides which requests are logged and at which level:

| Level | Logs |
|---|---|
| `None` | nothing |
| `Metadata` | user, verb, resource, time, response code (no body) |
| `Request` | + request body |
| `RequestResponse` | + response body |

Rule: log Secrets only at `Metadata` level, so secret values never land in log files.

On kind, audit logging must be set when the cluster is **created** (API server flags). So here we only write and read a policy. On OKE, API audit events go to **OCI Audit** / Logging.

In [ ]:
%%writefile labs/08/audit-policy.yaml
apiVersion: audit.k8s.io/v1
kind: Policy
omitStages: ["RequestReceived"]        # log once per request, at response time
rules:
  - level: None                        # skip noisy read-only health traffic
    nonResourceURLs: ["/healthz*", "/readyz*", "/livez*", "/version"]
  - level: Metadata                    # secrets: who touched them, but never the content
    resources:
      - group: ""
        resources: ["secrets", "configmaps"]
  - level: RequestResponse             # full detail for RBAC changes
    resources:
      - group: "rbac.authorization.k8s.io"
  - level: Request                     # body of exec / attach / port-forward requests
    resources:
      - group: ""
        resources: ["pods/exec", "pods/attach", "pods/portforward"]
  - level: Metadata                    # everything else: metadata only
    omitStages: ["RequestReceived"]

To use it on a kubeadm cluster you add API server flags: `--audit-policy-file=/etc/kubernetes/audit-policy.yaml`, `--audit-log-path=/var/log/kubernetes/audit.log`, `--audit-log-maxage=30` and mount the files into the API server static Pod.

### Hands-on lab: RBAC audit — who is cluster-admin?
This lists every subject bound to the `cluster-admin` ClusterRole. In production this list must be very short.

In [ ]:
%%bash
kubectl get clusterrolebindings -o json | python3 -c "
import json, sys
for b in json.load(sys.stdin)['items']:
    if b['roleRef']['name'] == 'cluster-admin':
        for s in b.get('subjects', []):
            print(f\"{b['metadata']['name']:40} {s['kind']:15} {s.get('namespace','-'):12} {s['name']}\")
"

**Expected output (kind)**
```
cluster-admin                            Group           -            system:masters
kubeadm:cluster-admins                   Group           -            kubeadm:cluster-admins
```

### Find ServiceAccounts that can read secrets in a namespace
A small loop: for each SA, ask `can-i`. In real clusters tools like `rbac-tool` or `kubectl-who-can` do this faster.

In [ ]:
%%bash
for sa in $(kubectl get sa -n lab-08 -o jsonpath='{.items[*].metadata.name}'); do
  printf "%-12s list secrets: %-4s  create pods: %-4s  exec: %s\n" "$sa" \
    "$(kubectl auth can-i list secrets -n lab-08 --as=system:serviceaccount:lab-08:$sa)" \
    "$(kubectl auth can-i create pods -n lab-08 --as=system:serviceaccount:lab-08:$sa)" \
    "$(kubectl auth can-i create pods/exec -n lab-08 --as=system:serviceaccount:lab-08:$sa)"
done

**Expected output**
```
default      list secrets: no    create pods: no    exec: no
deployer     list secrets: yes   create pods: yes   exec: yes     <- 'edit' is powerful!
viewer       list secrets: no    create pods: no    exec: no
```
Lesson: the built-in `edit` role can read secrets and exec into Pods. For CI deployers, write a tighter custom Role.

### Hands-on lab (optional): kube-bench CIS scan
kube-bench runs as a Job that reads node files. It needs internet to pull the image.

In [ ]:
%%bash
kubectl apply -n lab-08 -f https://raw.githubusercontent.com/aquasecurity/kube-bench/main/job.yaml >/dev/null
kubectl wait --for=condition=complete job/kube-bench -n lab-08 --timeout=180s
kubectl logs -n lab-08 job/kube-bench | grep -E "^\[(FAIL|WARN)\]|== Summary" | head -20

**Expected output (shape)**
```
[FAIL] 4.1.1 Ensure that the kubelet service file permissions are set to 600 or more restrictive
[WARN] 4.2.x ...
== Summary node ==
xx checks PASS
x checks FAIL
xx checks WARN
```
Each finding has an ID and remediation text. On managed clusters many control-plane checks do not apply; use the provider's benchmark profile (e.g. `--benchmark eks-1.x`, or CIS OKE benchmark guidance).

### Lab vs Production

| Area | Lab | Production |
|---|---|---|
| Audit log | Policy file only | Enabled; shipped to SIEM / OCI Logging; retained per compliance |
| RBAC review | Ad hoc commands | Scheduled report; alert on new cluster-admin bindings |
| CIS scan | One kube-bench run | Regular scans (kube-bench, Trivy Operator, Kubescape); findings tracked |
| Evidence | — | Reports kept for audits (ISO 27001, NCA ECC in KSA) |

### Check yourself
1. Why log Secrets only at `Metadata` level?
2. What does kube-bench check?
3. Which command lists all permissions of an identity?

<details><summary>Answer</summary>

1. Higher levels would write secret values into the log file.
2. Node and control-plane configuration against the CIS Kubernetes Benchmark.
3. `kubectl auth can-i --list --as=<identity> -n <ns>`.
</details>

---
# Interview and real-world questions

1. Explain the path of a `kubectl apply` through authentication, authorization and admission.
2. A developer says "I have edit rights but cannot create a RoleBinding." Why?
3. How do you give a Pod on OKE access to OCI Object Storage without API keys?
4. What replaced PodSecurityPolicy?
5. How would you prevent images from Docker Hub in production?
6. A certificate expired and broke an Ingress. How do you prevent this next time?
7. What is the risk of giving a CI pipeline cluster-admin?

<details><summary>Answer</summary>

1. Authentication (cert/token/OIDC) identifies the user. RBAC authorization checks verb/resource/namespace. Mutating admission may change the object, then schema validation, then validating admission (PSA, VAP, webhooks). Then the object is saved to etcd.
2. The `edit` ClusterRole has no permissions on RBAC objects, to prevent privilege escalation.
3. OKE Workload Identity: an OCI IAM policy allows the specific cluster + namespace + ServiceAccount to use the service; the Pod uses the OCI SDK with the workload identity provider.
4. Pod Security Admission with Pod Security Standards (privileged, baseline, restricted). Policy engines for extra rules.
5. An admission policy (VAP, Kyverno, Gatekeeper) that allows only images from the private registry (e.g. OCIR region endpoint), plus registry mirroring.
6. Manage certs with cert-manager (auto renewal), alert on expiry metrics, and keep an inventory.
7. If the pipeline or its token is stolen, the attacker owns the whole cluster. Use a namespaced Role with only needed verbs, short-lived tokens, and GitOps pull-based deploys.
</details>

---
# Practical Assignment — secure a namespace for a new team

**Scenario:** Team "orders" gets namespace `orders-lab`. You are the platform engineer.

**Requirements**
1. Namespace enforces Pod Security `restricted`.
2. ServiceAccount `orders-ci` can create/update Deployments, Services and ConfigMaps in `orders-lab` only. It cannot read Secrets and cannot exec into Pods.
3. ServiceAccount `orders-app` is used by the app. No token mounted.
4. Default-deny NetworkPolicy + DNS egress allowed.
5. Deploy `nginxinc/nginx-unprivileged:1.27-alpine` (listens on 8080) as Deployment `orders-web` with 2 replicas that passes `restricted`, with a read-only root filesystem.

**Success criteria**
- `kubectl auth can-i create deployments -n orders-lab --as=system:serviceaccount:orders-lab:orders-ci` → yes
- `kubectl auth can-i get secrets -n orders-lab --as=...:orders-ci` → no
- `kubectl auth can-i create pods/exec -n orders-lab --as=...:orders-ci` → no
- `kubectl get deploy orders-web -n orders-lab` shows 2/2 ready.

**Hints**
- nginx-unprivileged needs writable `/tmp` (emptyDir) when the root filesystem is read-only.
- Write your own Role. Do not use `edit`.

<details><summary>Full solution</summary>

```yaml
apiVersion: v1
kind: Namespace
metadata:
  name: orders-lab
  labels:
    pod-security.kubernetes.io/enforce: restricted
    pod-security.kubernetes.io/warn: restricted
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: orders-ci, namespace: orders-lab }
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: orders-app, namespace: orders-lab }
automountServiceAccountToken: false
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata: { name: orders-deployer, namespace: orders-lab }
rules:
  - apiGroups: ["apps"]
    resources: ["deployments"]
    verbs: ["get", "list", "watch", "create", "update", "patch"]
  - apiGroups: [""]
    resources: ["services", "configmaps"]
    verbs: ["get", "list", "watch", "create", "update", "patch"]
  - apiGroups: [""]
    resources: ["pods"]
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata: { name: orders-ci-deployer, namespace: orders-lab }
subjects:
  - { kind: ServiceAccount, name: orders-ci, namespace: orders-lab }
roleRef: { apiGroup: rbac.authorization.k8s.io, kind: Role, name: orders-deployer }
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata: { name: default-deny-all, namespace: orders-lab }
spec:
  podSelector: {}
  policyTypes: ["Ingress", "Egress"]
---
apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata: { name: allow-dns, namespace: orders-lab }
spec:
  podSelector: {}
  policyTypes: ["Egress"]
  egress:
    - to:
        - namespaceSelector: { matchLabels: { kubernetes.io/metadata.name: kube-system } }
          podSelector: { matchLabels: { k8s-app: kube-dns } }
      ports: [ { protocol: UDP, port: 53 }, { protocol: TCP, port: 53 } ]
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: orders-web, namespace: orders-lab }
spec:
  replicas: 2
  selector: { matchLabels: { app: orders-web } }
  template:
    metadata: { labels: { app: orders-web } }
    spec:
      serviceAccountName: orders-app
      automountServiceAccountToken: false
      securityContext:
        runAsNonRoot: true
        runAsUser: 101
        seccompProfile: { type: RuntimeDefault }
      containers:
        - name: web
          image: nginxinc/nginx-unprivileged:1.27-alpine
          ports: [ { containerPort: 8080 } ]
          securityContext:
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: true
            capabilities: { drop: ["ALL"] }
          resources:
            requests: { cpu: 50m, memory: 32Mi }
            limits: { memory: 64Mi }
          volumeMounts: [ { name: tmp, mountPath: /tmp } ]
      volumes: [ { name: tmp, emptyDir: {} } ]
```
Apply with `kubectl apply -f <file>`, then run the success-criteria commands. Clean up: `kubectl delete ns orders-lab`.
</details>

---
# "Break it" challenge

Run the cell below. It creates a Deployment in the `restricted` namespace. It will **not** become ready. Find out why **without reading the YAML first**. Use only `kubectl get`, `describe` and `events`.

In [ ]:
%%bash
kubectl apply -n lab-08-restricted -f - <<'EOF'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mystery
spec:
  replicas: 1
  selector: { matchLabels: { app: mystery } }
  template:
    metadata: { labels: { app: mystery } }
    spec:
      securityContext:
        seccompProfile: { type: RuntimeDefault }
      containers:
        - name: web
          image: nginx:1.27
          securityContext:
            allowPrivilegeEscalation: false
            capabilities: { drop: ["ALL"] }
            runAsNonRoot: true
EOF
sleep 10
kubectl get deploy,rs,pods -n lab-08-restricted -l app=mystery

<details><summary>Solution</summary>

`kubectl describe pod -l app=mystery -n lab-08-restricted` shows `CreateContainerConfigError` with
`container has runAsNonRoot and image will run as root`.
PSA allowed the Pod (it sets `runAsNonRoot: true`), but the kubelet refuses to start it because the `nginx` image runs as root (UID 0) and no `runAsUser` is set.

Fix options:
- Use `nginxinc/nginx-unprivileged:1.27-alpine` (runs as UID 101, port 8080), plus an `emptyDir` on `/tmp` if you add read-only root fs; or
- Set `runAsUser: 101` — but plain `nginx` then fails to bind port 80 and to write its cache dirs. The unprivileged image is the right fix.

Lesson: PSA checks the **spec**. The kubelet checks the **image** at start. Errors can come from both.
</details>

---
## Cleanup
Deletes everything this notebook created. cert-manager is cluster-wide; we remove it too (notebook 14 installs it again when needed).

In [ ]:
%%bash
kubectl delete namespace lab-08 lab-08-restricted --ignore-not-found --wait=false
kubectl delete clusterrolebinding viewer-reads-nodes --ignore-not-found
kubectl delete clusterrole node-reader --ignore-not-found
kubectl delete validatingadmissionpolicybinding no-latest-tag-lab --ignore-not-found
kubectl delete validatingadmissionpolicy no-latest-tag --ignore-not-found
kubectl delete -f https://github.com/cert-manager/cert-manager/releases/download/v1.15.3/cert-manager.yaml --ignore-not-found >/dev/null 2>&1
echo "Cleanup started. Namespaces may take a minute to disappear."

---
**Next notebook:** [09-Helm.ipynb](09-Helm.ipynb) — package and deploy apps with Helm.